# Analyze SINDy Autoencoder - Lorenz

Loads a trained checkpoint and checks reconstruction / SINDy error, matching the original `analyze_lorenz_model*.ipynb` notebooks.

In [ ]:
import os, sys

# Path to the SindyAutoencoders_Torch folder (this repo). If you uploaded/cloned
# it into Colab's local disk or mounted it from Google Drive, set this to that
# location - e.g. '/content/SindyAutoencoders_Torch' or
# '/content/drive/MyDrive/SindyAutoencoders_Torch'.
REPO_ROOT = '/content/SindyAutoencoders_Torch'

if not os.path.isdir(REPO_ROOT):
    # fall back to running locally, from within this notebook's own example folder
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

EXAMPLE_DIR = os.path.join(REPO_ROOT, 'examples', 'lorenz')
os.chdir(EXAMPLE_DIR)
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
sys.path.insert(0, EXAMPLE_DIR)
print('Working directory:', os.getcwd())


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from data import get_lorenz_data, generate_lorenz_data
from model import SindyAutoencoder
from sindy_utils import sindy_simulate


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cpu':
    print('No GPU detected - in Colab, check Runtime > Change runtime type > GPU.')


In [ ]:
save_name = 'model1'  # change to whatever checkpoint you trained

ckpt = torch.load(save_name + '.pt', map_location=device, weights_only=False)
params = ckpt['params']
model = SindyAutoencoder(params).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

def evaluate(data):
    batch = {k: torch.as_tensor(data[k], dtype=torch.float32, device=device)
             for k in ('x', 'dx', 'ddx') if k in data}
    with torch.no_grad():
        out = model(batch)
    return {k: v.detach().cpu().numpy() for k, v in out.items()}


## Single trajectory: identified vs. true dynamics

In [ ]:
t = np.arange(0, 20, .01)
z0 = np.array([[-8, 7, 27]])
test_data = generate_lorenz_data(z0, t, params['input_dim'], linear=False,
                                  normalization=np.array([1/40, 1/40, 1/40]))
test_data['x'] = test_data['x'].reshape((-1, params['input_dim']))
test_data['dx'] = test_data['dx'].reshape((-1, params['input_dim']))
test_data['z'] = test_data['z'].reshape((-1, params['latent_dim']))

results = evaluate(test_data)


In [ ]:
z_sim = sindy_simulate(results['z'][0], t,
                       params['coefficient_mask']*results['sindy_coefficients'],
                       params['poly_order'], params['include_sine'])
lorenz_sim = sindy_simulate(test_data['z'][0], t, test_data['sindy_coefficients'],
                            params['poly_order'], params['include_sine'])

fig = plt.figure(figsize=(8, 4))
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(z_sim[:, 0], z_sim[:, 1], z_sim[:, 2], linewidth=1)
ax1.set_title('Identified model')
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(lorenz_sim[:, 0], lorenz_sim[:, 1], lorenz_sim[:, 2], linewidth=1, color='tab:orange')
ax2.set_title('True Lorenz')
plt.show()


In [ ]:
plt.figure(figsize=(8, 2))
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.plot(t, results['z'][:, i], color='#888888', linewidth=2, label='encoder z')
    plt.plot(t, z_sim[:, i], '--', linewidth=2, label='SINDy-simulated')
    if i == 0:
        plt.legend(fontsize=7)
plt.tight_layout()
plt.show()


## Identified SINDy coefficients vs. ground truth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(4, 3))
Xi_plot = params['coefficient_mask']*results['sindy_coefficients']
axes[0].imshow(Xi_plot, aspect='auto'); axes[0].set_title('identified')
axes[1].imshow(test_data['sindy_coefficients'], aspect='auto'); axes[1].set_title('true')
plt.tight_layout()
plt.show()


## Held-out trajectories: reconstruction / SINDy error

In [ ]:
test_data = get_lorenz_data(100, noise_strength=1e-6)
results = evaluate(test_data)

decoder_x_error = np.mean((test_data['x'] - results['x_decode'])**2)/np.mean(test_data['x']**2)
decoder_dx_error = np.mean((test_data['dx'] - results['dx_decode'])**2)/np.mean(test_data['dx']**2)
sindy_dz_error = np.mean((results['dz'] - results['dz_predict'])**2)/np.mean(results['dz']**2)

print('Decoder relative error: %f' % decoder_x_error)
print('Decoder relative SINDy error: %f' % decoder_dx_error)
print('SINDy relative error, z: %f' % sindy_dz_error)
